# Part I: Generative AI - Encoder-Decoder Attention in BART Summarisation

## Learning Outcome
Explain the fundamentals of Generative AI and various large language models (LLMs) based on their architectures, capabilities and applications.

---

## Assignment Overview
In this assignment, we will fine-tune a BART model to summarise news articles from the CNN/DailyMail dataset. The focus is on understanding how encoder-decoder attention helps extract relevant content from long texts and generate coherent summaries.

### Tasks:
1. Load and preprocess the CNN/DailyMail dataset
2. Fine-tune a pre-trained encoder-decoder model (facebook/bart-base)
3. Evaluate summarisation quality using ROUGE scores
4. Show article-summary pairs and analyse how attention contributed to key content extraction
5. Explain how encoder-decoder attention differs from self-attention and its interpretability

## Step 1: Install Required Libraries

First, we need to install the necessary libraries for this assignment.

In [ ]:
!pip install transformers datasets tensorflow tensorflow_datasets rouge-score sentencepiece

**Important:** After running the above cell, restart the runtime/session before continuing.

In Google Colab: Runtime -> Restart runtime

## Step 2: Import Libraries

In [ ]:
import tensorflow_datasets as tfds
from datasets import Dataset
import pandas as pd
import torch
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from rouge_score import rouge_scorer
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 3: Load and Preprocess the CNN/DailyMail Dataset

We will load a subset of the CNN/DailyMail dataset for training and testing. This dataset contains news articles paired with human-written summaries (highlights).

In [ ]:
# Load the CNN/DailyMail dataset using TensorFlow Datasets
print("Loading training data...")
ds_train_tf = tfds.load('cnn_dailymail:3.4.0', split='train[:1%]', shuffle_files=True)

print("Loading test data...")
ds_test_tf = tfds.load('cnn_dailymail:3.4.0', split='test[:1%]', shuffle_files=True)

print("Dataset loaded successfully!")

In [ ]:
# Extract articles and highlights from training data
train_articles, train_highlights = [], []
for item in ds_train_tf.take(2000):  # Using up to 2000 samples for training
    train_articles.append(item['article'].numpy().decode())
    train_highlights.append(item['highlights'].numpy().decode())

# Extract articles and highlights from test data
test_articles, test_highlights = [], []
for item in ds_test_tf.take(500):  # Using up to 500 samples for testing
    test_articles.append(item['article'].numpy().decode())
    test_highlights.append(item['highlights'].numpy().decode())

print(f"Training samples: {len(train_articles)}")
print(f"Test samples: {len(test_articles)}")

In [ ]:
# Convert to Hugging Face Dataset format
df_train = pd.DataFrame({'article': train_articles, 'highlights': train_highlights})
df_test = pd.DataFrame({'article': test_articles, 'highlights': test_highlights})

train_data = Dataset.from_pandas(df_train)
test_data = Dataset.from_pandas(df_test)

print("\nDataset structure:")
print(train_data)

In [ ]:
# Display a sample article and its summary
print("=" * 80)
print("SAMPLE ARTICLE:")
print("=" * 80)
print(train_articles[0][:1500] + "...")
print("\n" + "=" * 80)
print("REFERENCE SUMMARY (HIGHLIGHTS):")
print("=" * 80)
print(train_highlights[0])

## Step 4: Tokenization and Data Preprocessing

We will use the BART tokenizer to convert text into tokens that the model can process. BART uses a byte-pair encoding (BPE) tokenizer.

In [ ]:
# Load the BART tokenizer and model
model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

print(f"Model: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Define preprocessing function
def preprocess_function(examples):
    """
    Tokenize articles (input) and highlights (target summaries).
    
    - max_input_length: Maximum length for source articles (truncated if longer)
    - max_target_length: Maximum length for target summaries
    """
    max_input_length = 512  # BART's maximum input length
    max_target_length = 128  # Maximum summary length
    
    # Tokenize the articles (encoder input)
    model_inputs = tokenizer(
        examples['article'],
        max_length=max_input_length,
        truncation=True,
        padding='max_length'
    )
    
    # Tokenize the summaries (decoder target)
    labels = tokenizer(
        examples['highlights'],
        max_length=max_target_length,
        truncation=True,
        padding='max_length'
    )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

In [ ]:
# Apply preprocessing to the datasets
print("Tokenizing training data...")
tokenized_train = train_data.map(
    preprocess_function,
    batched=True,
    remove_columns=train_data.column_names
)

print("Tokenizing test data...")
tokenized_test = test_data.map(
    preprocess_function,
    batched=True,
    remove_columns=test_data.column_names
)

print("\nTokenized dataset structure:")
print(tokenized_train)

## Step 5: Fine-tune the BART Model

Now we will fine-tune the pre-trained BART model on our CNN/DailyMail dataset. Fine-tuning adapts the model's weights to perform better on the specific task of news summarisation.

In [ ]:
# Create data collator for sequence-to-sequence tasks
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [ ]:
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-summarization",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,  # Adjust based on GPU memory
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,  # Number of training epochs
    predict_with_generate=True,
    logging_dir='./logs',
    logging_steps=100,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
    report_to="none"  # Disable wandb logging
)

print("Training configuration:")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - Mixed precision (fp16): {training_args.fp16}")

In [ ]:
# Initialize the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("Trainer initialized successfully!")

In [ ]:
# Fine-tune the model
print("Starting fine-tuning...")
print("This may take a while depending on your hardware.\n")

trainer.train()

print("\nFine-tuning completed!")

In [ ]:
# Save the fine-tuned model
model.save_pretrained("./bart-summarization-finetuned")
tokenizer.save_pretrained("./bart-summarization-finetuned")
print("Model saved to ./bart-summarization-finetuned")

## Step 6: Generate Summaries and Evaluate with ROUGE Scores

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a set of metrics used to evaluate text summarisation:

- **ROUGE-1**: Measures unigram (single word) overlap
- **ROUGE-2**: Measures bigram (two consecutive words) overlap  
- **ROUGE-L**: Measures longest common subsequence

In [ ]:
def generate_summary(article, model, tokenizer, max_length=128):
    """
    Generate a summary for a given article using the fine-tuned BART model.
    """
    # Tokenize the input article
    inputs = tokenizer(
        article,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Move model to device
    model.to(device)
    
    # Generate summary
    summary_ids = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=max_length,
        num_beams=4,  # Beam search for better quality
        length_penalty=2.0,
        early_stopping=True,
        no_repeat_ngram_size=3  # Avoid repetition
    )
    
    # Decode the generated summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
def calculate_rouge_scores(predictions, references):
    """
    Calculate ROUGE scores for generated summaries against reference summaries.
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)
    
    return {
        'rouge1': np.mean(rouge1_scores),
        'rouge2': np.mean(rouge2_scores),
        'rougeL': np.mean(rougeL_scores)
    }

In [ ]:
# Evaluate on a subset of test data
num_eval_samples = 50  # Evaluate on 50 samples
print(f"Generating summaries for {num_eval_samples} test samples...\n")

generated_summaries = []
reference_summaries = test_highlights[:num_eval_samples]

for i, article in enumerate(test_articles[:num_eval_samples]):
    summary = generate_summary(article, model, tokenizer)
    generated_summaries.append(summary)
    
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{num_eval_samples} samples")

print("\nSummary generation complete!")

In [ ]:
# Calculate ROUGE scores
rouge_scores = calculate_rouge_scores(generated_summaries, reference_summaries)

print("=" * 60)
print("ROUGE EVALUATION SCORES")
print("=" * 60)
print(f"ROUGE-1 (Unigram Overlap):     {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2 (Bigram Overlap):      {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L (Longest Subsequence): {rouge_scores['rougeL']:.4f}")
print("=" * 60)

## Step 7: Display Article-Summary Pairs and Attention Analysis

Let's examine some examples of generated summaries and analyse how attention mechanisms contribute to content extraction.

In [ ]:
def display_summary_comparison(article, reference, generated, index):
    """
    Display a comparison between original article, reference summary, and generated summary.
    """
    print("\n" + "=" * 80)
    print(f"EXAMPLE {index + 1}")
    print("=" * 80)
    
    print("\n--- ORIGINAL ARTICLE (truncated) ---")
    print(article[:1000] + "..." if len(article) > 1000 else article)
    
    print("\n--- REFERENCE SUMMARY (Human-written) ---")
    print(reference)
    
    print("\n--- GENERATED SUMMARY (BART) ---")
    print(generated)
    
    # Calculate individual ROUGE scores
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    
    print("\n--- ROUGE SCORES FOR THIS EXAMPLE ---")
    print(f"ROUGE-1: {scores['rouge1'].fmeasure:.4f}")
    print(f"ROUGE-2: {scores['rouge2'].fmeasure:.4f}")
    print(f"ROUGE-L: {scores['rougeL'].fmeasure:.4f}")

In [ ]:
# Display 3 example summaries
print("ARTICLE-SUMMARY PAIR ANALYSIS")
print("Examining how BART's encoder-decoder attention extracts key content\n")

for i in range(3):
    display_summary_comparison(
        test_articles[i],
        reference_summaries[i],
        generated_summaries[i],
        i
    )

---

## Question 1: How does encoder-decoder attention help in generating high-quality summaries in models like BART?

### Answer:

Encoder-decoder attention (also known as **cross-attention**) is a critical mechanism that enables BART to generate high-quality summaries by establishing a dynamic connection between the input article and the generated output. Here's how it contributes:

#### 1. **Selective Information Extraction**
- The encoder processes the entire input article and creates contextual representations for each token
- During decoding, cross-attention allows each generated word to "look at" all positions in the encoded input
- The attention mechanism assigns higher weights to relevant parts of the source text, effectively extracting key information
- For example, when generating a word about a person's name, the model attends heavily to the tokens where that name appears in the article

#### 2. **Maintaining Factual Consistency**
- Cross-attention provides a direct pathway to the original content
- This helps prevent hallucination by grounding generated text in the source document
- The model can "copy" or paraphrase information from attended positions rather than generating from scratch

#### 3. **Handling Long Documents**
- Unlike simple sequence-to-sequence models, encoder-decoder attention can selectively focus on distant parts of long documents
- The attention mechanism creates shortcuts that bypass the sequential processing limitation
- Different decoder positions can attend to different parts of the article as needed

#### 4. **Generating Coherent Multi-Sentence Summaries**
- As the decoder generates each token, it uses cross-attention to gather relevant context
- This allows the model to:
  - Maintain topic focus throughout the summary
  - Correctly order information based on importance
  - Generate grammatically correct text that accurately reflects the source

#### 5. **Soft Alignment Learning**
- Unlike hard extraction methods, attention learns soft alignments during training
- The model automatically learns what constitutes "important" information for summarisation
- This allows for abstraction and paraphrasing while maintaining semantic accuracy

#### Mathematical Formulation:
```
Cross-Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

Where:
- Q (Query): comes from the decoder's hidden states
- K (Key): comes from the encoder's output
- V (Value): comes from the encoder's output
- d_k: dimension of the key vectors
```

This mechanism allows each decoder position to compute a weighted sum of encoder representations, where the weights are determined by the relevance of each source position to the current generation step.

---

## Question 2: How does encoder-decoder attention differ from self-attention, and why is this distinction important for summarisation?

### Answer:

#### Key Differences Between Encoder-Decoder Attention and Self-Attention:

| Aspect | Self-Attention | Encoder-Decoder (Cross) Attention |
|--------|---------------|----------------------------------|
| **Source of Q, K, V** | All from the same sequence | Q from decoder, K and V from encoder |
| **Purpose** | Learn relationships within a single sequence | Learn relationships between two different sequences |
| **Location** | Encoder layers and decoder layers | Only in decoder layers |
| **What it learns** | Contextual representations of input/output | Alignment between source and target |

#### Detailed Comparison:

**1. Self-Attention (Intra-sequence attention)**
```
- In Encoder: Each input token attends to all other input tokens
- In Decoder: Each output token attends to previous output tokens (masked)
- Query, Key, Value all come from the SAME sequence
- Learns contextual relationships WITHIN a sequence
```

**2. Encoder-Decoder Attention (Cross-attention)**
```
- Query comes from the DECODER (what we're generating)
- Key and Value come from the ENCODER (the source article)
- Creates a BRIDGE between input and output sequences
- Learns alignment and relevance ACROSS sequences
```

#### Why This Distinction is Critical for Summarisation:

**1. Separation of Concerns**
- **Self-attention in encoder**: Builds rich, contextual understanding of the article
  - Resolves pronouns ("he" refers to "John Smith")
  - Captures long-range dependencies
  - Creates holistic document representation
  
- **Self-attention in decoder**: Ensures generated summary is coherent
  - Maintains grammatical structure
  - Avoids repetition
  - Keeps track of what has already been said

- **Cross-attention**: Connects the two sequences
  - Decides WHAT information from the article to include
  - Ensures faithfulness to source
  - Enables content selection and compression

**2. Interpretability Benefits**
- Cross-attention weights can be visualised to understand model decisions
- We can see which parts of the article influenced each generated word
- This provides transparency in the summarisation process
- Self-attention patterns show internal reasoning but are harder to interpret for summarisation quality

**3. Enabling Abstractive Summarisation**
- Self-attention alone (decoder-only models) struggles with long document summarisation
- Cross-attention allows:
  - Efficient processing of long articles (encoded once, accessed multiple times)
  - Flexible content selection regardless of position
  - True abstraction while maintaining factual grounding

**4. Preventing Information Loss**
- Without cross-attention, the entire article would need to be compressed into a fixed representation
- Cross-attention provides direct access to all encoder positions
- No bottleneck - the full source representation is available during generation

#### Visual Representation:

```
BART Architecture for Summarisation:

[Article Tokens] --> ENCODER (Self-Attention) --> [Contextual Representations]
                                                         |
                                                         | Cross-Attention
                                                         v
[Start Token] --> DECODER (Masked Self-Attention + Cross-Attention) --> [Summary Tokens]
```

In summary, the distinction between these attention types creates a modular architecture where:
- The encoder specialises in understanding the input
- The decoder specialises in generating fluent text
- Cross-attention specialises in bridging these two tasks

This division of labor is what makes encoder-decoder models like BART particularly effective for summarisation tasks.

---

## Summary and Conclusions

In this assignment, we have:

1. **Loaded and preprocessed** the CNN/DailyMail dataset for news summarisation

2. **Fine-tuned** a pre-trained BART model (`facebook/bart-base`) on the summarisation task

3. **Evaluated** summarisation quality using ROUGE metrics:
   - ROUGE-1: Measures unigram overlap
   - ROUGE-2: Measures bigram overlap
   - ROUGE-L: Measures longest common subsequence

4. **Analysed** article-summary pairs to understand how the model extracts key content and generates coherent summaries

5. **Explained** the fundamental differences between encoder-decoder attention and self-attention, and why this distinction is crucial for summarisation tasks

### Key Takeaways:

- **Encoder-decoder attention** is the bridge that allows summarisation models to selectively extract and compress information from long documents
- **Self-attention** builds contextual understanding within sequences, while **cross-attention** enables information flow between sequences
- **BART's bidirectional encoder** captures full context of the input, while its **autoregressive decoder** generates coherent summaries
- **Attention mechanisms** provide interpretability, allowing us to understand which parts of the source document influenced each generated word
- **Fine-tuning** pre-trained models is an effective approach for domain-specific summarisation tasks

---

## References

1. Lewis, M., et al. (2020). BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension. *ACL 2020*.

2. Vaswani, A., et al. (2017). Attention Is All You Need. *NeurIPS 2017*.

3. Hermann, K.M., et al. (2015). Teaching Machines to Read and Comprehend. *NeurIPS 2015*. (CNN/DailyMail Dataset)

4. Lin, C.Y. (2004). ROUGE: A Package for Automatic Evaluation of Summaries. *ACL Workshop on Text Summarization*.

5. Hugging Face Transformers Documentation: https://huggingface.co/docs/transformers/